# Spec-Align: DPO Fine-Tuning on TLA+ Preference Pairs

**Spec-Align** fine-tunes a base LLM using **Direct Preference Optimization (DPO)** on the TLA+Bench **PVP dataset**.

Each preference pair consists of:
- **prompt** — the real natural language description from `TLA_Description.json`
- **chosen** — a generated TLA+ spec that passes SANY syntax checking
- **rejected** — a generated TLA+ spec that fails SANY syntax checking

Pairs are generated from the **train split (124 specs)** only. The **test split (41 specs)** is held out for evaluation.

| | |
|---|---|
| **Base model** | `deepseek-ai/deepseek-coder-6.7b-base` (fully cached, code-focused) |
| **Preference pairs** | Generated from train split via `deepseek-r1:8b` + SANY validation |
| **Condition** | `nlp_to_tla` — real English descriptions as prompts |
| **Adaptation** | LoRA on q/k/v/o attention projections |
| **Implementation** | `src/dpo_trainer.py` |

> **Kernel:** Use the **Python 3 (ipykernel)** kernel — it has PyTorch 2.7 with CUDA support.

## Section 1: Imports and Environment

In [1]:
import json
import sys
import os
from pathlib import Path

import torch
import numpy as np

# ipywidgets 6.x is installed but transformers expects 7.x+.
# Disabling the progress bar prevents the FloatProgress.style AttributeError.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import transformers
transformers.logging.disable_progress_bar()

# Add src/ to path so we can import from dpo_trainer.py
REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: No CUDA — switch to the Python 3 (ipykernel) kernel and rerun.")

Python:       3.10.12
PyTorch:      2.7.0
Transformers: 4.45.0
CUDA:         True
  GPU 0: Quadro RTX 8000  50.8 GB
  GPU 1: Quadro RTX 8000  50.7 GB


## Section 2: Load and Inspect the Preference Dataset

In [2]:
DATASET_PATH = REPO_ROOT / "outputs" / "preference_dataset.jsonl"

pairs = []
with open(DATASET_PATH) as f:
    for line in f:
        if line.strip():
            pairs.append(json.loads(line))

print(f"Total preference pairs : {len(pairs)}")
print(f"Keys in each record    : {list(pairs[0].keys())}")

Total preference pairs : 123
Keys in each record    : ['prompt', 'chosen', 'rejected', 'spec_id']


In [3]:
print("=" * 70)
print("DATASET STATISTICS")
print("=" * 70)

prompt_lens   = [len(p["prompt"])   for p in pairs]
chosen_lens   = [len(p["chosen"])   for p in pairs]
rejected_lens = [len(p["rejected"]) for p in pairs]

for label, lens in [("prompt", prompt_lens), ("chosen", chosen_lens), ("rejected", rejected_lens)]:
    print(f"{label:>10}  min={min(lens):4d}  max={max(lens):4d}  mean={np.mean(lens):.0f} chars")

print("=" * 70)
print("\nSample pair (index 0):")
print(f"  prompt   : {pairs[0]['prompt'][:120]}")
print(f"  chosen   : {pairs[0]['chosen'][:120]}")
print(f"  rejected : {pairs[0]['rejected'][:120]}")

DATASET STATISTICS
    prompt  min= 121  max= 196  mean=146 chars
    chosen  min=  82  max= 144  mean=105 chars
  rejected  min=  57  max= 117  mean=79 chars

Sample pair (index 0):
  prompt   : Create a TLA+ specification for module ACP_NB. The module should include operators: TypeInvParticipantNB, TypeInvNB, Ini
  chosen   : ---- MODULE ACP_NB ----
EXTENDS Naturals, Sequences

Init == TRUE

Next == UNCHANGED <<>>

Spec == Init /\ [][Next]_<<>>
  rejected : ---- MODULE ACP_NB ----
EXTENDS Naturals

Init = TRUE

Next == UNCHANGED

Spec == Init /\ Next
====


## Section 3: Create Train / Eval Splits (80 / 20)

In [4]:
TRAIN_PATH = REPO_ROOT / "outputs" / "preference_train.jsonl"
EVAL_PATH  = REPO_ROOT / "outputs" / "preference_eval.jsonl"

split_idx   = int(0.8 * len(pairs))
train_pairs = pairs[:split_idx]
eval_pairs  = pairs[split_idx:]

for path, subset in [(TRAIN_PATH, train_pairs), (EVAL_PATH, eval_pairs)]:
    with open(path, "w") as f:
        for p in subset:
            f.write(json.dumps(p) + "\n")

print(f"Train: {len(train_pairs)} pairs  ->  {TRAIN_PATH}")
print(f"Eval:  {len(eval_pairs)}  pairs  ->  {EVAL_PATH}")

Train: 98 pairs  ->  /home/arslanbisharat/Documents/AI4FM/tla bench dashboard/outputs/preference_train.jsonl
Eval:  25  pairs  ->  /home/arslanbisharat/Documents/AI4FM/tla bench dashboard/outputs/preference_eval.jsonl


## Section 4: Configure Spec-Align (DPO) Training

In [ ]:
# deepseek-coder-6.7b-base is fully cached locally (13GB) and code-focused — ideal for TLA+.
# DeepSeek-R1-Distill-Llama-8B is NOT fully cached and would trigger a 16GB download.
BASE_MODEL = "deepseek-ai/deepseek-coder-6.7b-base"
OUTPUT_DIR = REPO_ROOT / "outputs" / "dpo_models"

DPO_CONFIG = {
    "beta":          0.1,
    "learning_rate": 1e-4,
    "num_epochs":    3,
    "batch_size":    4,
    "warmup_steps":  20,
    "max_length":    256,
    "use_lora":      True,
    "lora_rank":     16,
    "lora_alpha":    32,
    "lora_dropout":  0.05,
}

print("=" * 50)
print("SPEC-ALIGN DPO CONFIGURATION")
print("=" * 50)
print(f"Base model : {BASE_MODEL}")
print(f"Output dir : {OUTPUT_DIR}")
for k, v in DPO_CONFIG.items():
    print(f"  {k:<20} {v}")
print("=" * 50)

## Section 5: Initialize the DPO Trainer

This loads the base model **twice**:
1. **Policy model** — fine-tuned with LoRA (trainable)
2. **Reference model** — frozen deepcopy of the base model (provides KL baseline)

In [6]:
from dpo_trainer import DPOTrainer

trainer = DPOTrainer(
    model_name    = BASE_MODEL,
    output_dir    = str(OUTPUT_DIR),
    beta          = DPO_CONFIG["beta"],
    learning_rate = DPO_CONFIG["learning_rate"],
    num_epochs    = DPO_CONFIG["num_epochs"],
    batch_size    = DPO_CONFIG["batch_size"],
    warmup_steps  = DPO_CONFIG["warmup_steps"],
    max_length    = DPO_CONFIG["max_length"],
    use_lora      = DPO_CONFIG["use_lora"],
    lora_rank     = DPO_CONFIG["lora_rank"],
    lora_alpha    = DPO_CONFIG["lora_alpha"],
    lora_dropout  = DPO_CONFIG["lora_dropout"],
    wandb_project = "tla_bench",
    wandb_run_name= "spec-align-dpo",
)

total_params     = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)

print(f"\nDevice          : {trainer.device}")
print(f"Policy model    : {total_params:,} total  |  {trainable_params:,} trainable (LoRA)")
print(f"Reference model : {sum(p.numel() for p in trainer.ref_model.parameters()):,} params (frozen)")

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## Section 6: Run DPO Training

Metrics logged to W&B `tla_bench` project under group `dpo_training`:
- `train/loss` — DPO loss per batch
- `train/reward_margin` — chosen reward minus rejected reward (should rise)
- `eval/preference_accuracy` — fraction of eval pairs where policy prefers chosen over rejected

In [ ]:
train_history = trainer.train(
    train_dataset_path = str(TRAIN_PATH),
    eval_dataset_path  = str(EVAL_PATH),
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Epochs trained       : {DPO_CONFIG['num_epochs']}")
print(f"Final epoch loss     : {train_history['epoch_losses'][-1]:.4f}")
if "eval_results" in train_history:
    er = train_history["eval_results"]
    print(f"Eval loss            : {er['eval_loss']:.4f}")
    print(f"Preference accuracy  : {er['preference_accuracy']:.4f}")
print("=" * 60)

## Section 7: Training Curves

In [ ]:
import matplotlib.pyplot as plt

batch_losses     = train_history["batch_losses"]
chosen_rewards   = train_history["chosen_rewards"]
rejected_rewards = train_history["rejected_rewards"]
reward_margins   = [c - r for c, r in zip(chosen_rewards, rejected_rewards)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(batch_losses, linewidth=1)
axes[0].set_title("DPO Loss (per batch)")
axes[0].set_xlabel("Batch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(reward_margins, linewidth=1, color="green")
axes[1].axhline(0, color="red", linestyle="--", linewidth=0.8)
axes[1].set_title("Reward Margin: chosen - rejected (per batch)")
axes[1].set_xlabel("Batch")
axes[1].set_ylabel("Reward Margin")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Spec-Align DPO Training", fontsize=13, fontweight="bold")
plt.tight_layout()

fig_path = OUTPUT_DIR / "training_curves.png"
plt.savefig(str(fig_path), dpi=150)
plt.show()
print(f"Saved: {fig_path}")

## Section 8: Save Spec-Align Checkpoint

In [ ]:
checkpoint_path = trainer.save_checkpoint()

metadata = {
    "base_model"          : BASE_MODEL,
    "method"              : "DPO (Spec-Align)",
    "dataset_size"        : len(pairs),
    "train_pairs"         : len(train_pairs),
    "eval_pairs"          : len(eval_pairs),
    "config"              : DPO_CONFIG,
    "final_train_loss"    : train_history["epoch_losses"][-1],
    "eval_loss"           : train_history.get("eval_results", {}).get("eval_loss"),
    "eval_preference_acc" : train_history.get("eval_results", {}).get("preference_accuracy"),
    "checkpoint_path"     : checkpoint_path,
}

meta_path = OUTPUT_DIR / "spec_align_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Model checkpoint : {checkpoint_path}")
print(f"Metadata         : {meta_path}")
print("\nSpec-Align training complete.")